In [2]:
import numpy as np

def top_bottom_jaccard(scores_a, scores_b, frac=0.10):
    """
    scores_a, scores_b: 1D lists/arrays of length N (one score per row)
    frac: fraction in (0, 1], e.g. 0.10 for top/bottom 10%

    Returns: dict with top/bottom sets and their Jaccard indices.
    """
    a = np.asarray(scores_a).ravel()
    b = np.asarray(scores_b).ravel()
    assert a.shape == b.shape, "scores must have same length"
    n = a.shape[0]
    k = max(1, int(round(frac * n)))

    # indices of top-k
    top_a = set(np.argpartition(a, -k)[-k:].tolist())
    top_b = set(np.argpartition(b, -k)[-k:].tolist())

    # indices of bottom-k
    bot_a = set(np.argpartition(a,  k)[:k].tolist())
    bot_b = set(np.argpartition(b,  k)[:k].tolist())

    def jaccard(A, B):
        union = A | B
        return len(A & B) / len(union) if union else 0.0

    return jaccard(top_a, top_b), jaccard(bot_a, bot_b)


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from pathlib import Path
import torch
from scipy.stats import spearmanr
from tqdm import tqdm


diffmean_root = "../projection_pca_64"

hinge_path = "pca_64/projection_pca_hinge"
logistic_path = "pca_64/projection_pca_logistic"

frac = 0.2


diffmean_files = list(Path(diffmean_root).glob("*.pt"))


jaccard_real = []

for file in tqdm(diffmean_files):
    diffmean = torch.load(file)

    
    result = {
        'benchmark': diffmean['benchmark'],
        'model': diffmean['model_name'],
        'column': diffmean['column'],
        'frac': frac,
    }
    model = diffmean['model_name'].split("/")[-1]
    
    benchmark = diffmean['benchmark']
    
    dataset = pd.read_csv(f"../{benchmark}.csv")
    
    indices = dataset[dataset['dataset']=='test']['id'].tolist()
    indices = np.array(indices)
    del dataset
    

    if model == "Qwen3-30B-A3B-Instruct-2507":
        continue
    if diffmean['column'] == 'longer_context':
        continue

    file_name = f"{benchmark}_{diffmean['column']}_{model}"

    probe_file_name = f"{file_name}_projection.pt"

    hinge = torch.load(os.path.join(hinge_path, probe_file_name))
    logistic = torch.load(os.path.join(logistic_path, probe_file_name))

    N, num_layers = diffmean['projection'].shape
    
    
    diffmean_hinge_top = np.zeros((num_layers,))
    diffmean_hinge_bot = np.zeros((num_layers,))
    diffmean_logistic_top = np.zeros((num_layers,))
    diffmean_logistic_bot = np.zeros((num_layers,))
    hinge_logistic_top = np.zeros((num_layers,))
    hinge_logistic_bot = np.zeros((num_layers,))
    for layer_index in range(num_layers):
        diffmean_proj_test = diffmean['projection'][:, layer_index][indices].numpy()
        
        hinge_proj_test = hinge['projection'][:, layer_index][indices].numpy()
        logistic_proj_test = logistic['projection'][:, layer_index][indices].numpy()

        diffmean_hinge_top[layer_index], diffmean_hinge_bot[layer_index]= top_bottom_jaccard(diffmean_proj_test, hinge_proj_test, frac=frac)
        diffmean_logistic_top[layer_index], diffmean_logistic_bot[layer_index]= top_bottom_jaccard(diffmean_proj_test, logistic_proj_test, frac=frac)
        hinge_logistic_top[layer_index], hinge_logistic_bot[layer_index]= top_bottom_jaccard(hinge_proj_test, logistic_proj_test, frac=frac)

    result['diffmean_hinge_top'] = diffmean_hinge_top.tolist()
    result['diffmean_hinge_bot'] = diffmean_hinge_bot.tolist()
    result['diffmean_logistic_top'] = diffmean_logistic_top.tolist()
    result['diffmean_logistic_bot'] = diffmean_logistic_bot.tolist()
    result['hinge_logistic_top'] = hinge_logistic_top.tolist()
    result['hinge_logistic_bot'] = hinge_logistic_bot.tolist()

    jaccard_real.append(result)


with open("jaccard_real.json", "w") as f:
    for item in jaccard_real:
        f.write(json.dumps(item) + "\n")
        


100%|██████████| 30/30 [00:01<00:00, 20.48it/s]


In [16]:
import json
with open("jaccard_real.json", "r") as f:
    jaccard_real = [json.loads(line) for line in f.readlines()]

print(len(jaccard_real))

20
